# Gaussian-probe transforms and plots
Run from this notebook's directory, or set `script` to its full path. Each CSV is transformed with its specified frequency sign, then the complex transforms are summed. All inputs must have matching site and time grids and compatible prefactor conventions. No lesser/greater prefactor or sign conversion is applied automatically.

In [ ]:
from pathlib import Path
import csv
import subprocess
import sys
import tempfile
import numpy as np
import matplotlib.pyplot as plt

script = Path('fourier_probe_twotime.py')
mode = 'both'  # 'lesser', 'greater', or 'both'
lesser_csv = Path('lesser_correlator.csv')
greater_csv = Path('greater_correlator.csv')
output_csv = Path('fourier_probe_output.csv')
sigma = 1.0
omega_min, omega_max, omega_count = -5.0, 5.0, 101
probe_min, probe_max, probe_count = 0.0, 10.0, 51
input_convention = 'correlator'  # 'correlator' or 'green'; applies to every input
frequency_shift = 4.0  # Subtracted from frequency in plots only


## Transform and sum
Set `mode` above to compute only lesser, only greater, or both. The output CSV is overwritten. `lesser` uses exp(+iω(t−t′)); `greater` uses exp(−iω(t−t′)).

In [ ]:
if not script.is_file():
    raise FileNotFoundError(f'Transform script not found: {script.resolve()}')
if mode not in ('lesser', 'greater', 'both'):
    raise ValueError("mode must be 'lesser', 'greater', or 'both'")
inputs = []
if mode in ('lesser', 'both'):
    inputs.append((lesser_csv, 'lesser'))
if mode in ('greater', 'both'):
    inputs.append((greater_csv, 'greater'))
if any(output_csv.resolve() == path.resolve() for path, _ in inputs):
    raise ValueError('Output must not overwrite an input')
summed = {}
reference_keys = None
with tempfile.TemporaryDirectory() as tmp:
    for i, (path, kind) in enumerate(inputs):
        if kind not in ('lesser', 'greater'):
            raise ValueError(f'Unknown frequency sign: {kind}')
        transformed = Path(tmp) / f'transform_{i}.csv'
        command = [sys.executable, str(script), str(path), str(transformed),
                   '--sigma', str(sigma), '--omega-min', str(omega_min),
                   '--omega-max', str(omega_max), '--omega-count', str(omega_count),
                   '--probe-min', str(probe_min), '--probe-max', str(probe_max),
                   '--probe-count', str(probe_count), '--input-convention', input_convention,
                   '--frequency-sign', kind]
        subprocess.run(command, check=True)
        with transformed.open(newline='', encoding='utf-8') as f:
            data = {}
            for row in csv.DictReader(f):
                key = (int(row['center_site']), int(row['site_index']), float(row['probe_time']), float(row['omega']))
                if key in data:
                    raise ValueError(f'Duplicate transformed point: {key}')
                data[key] = complex(float(row['real_part']), float(row['imaginary_part']))
        if reference_keys is None:
            reference_keys = set(data)
        elif set(data) != reference_keys:
            raise ValueError(f'Transformed site/probe/frequency points do not match for {path}')
        for key, value in data.items():
            summed[key] = summed.get(key, 0j) + value
with output_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(('center_site', 'site_index', 'probe_time', 'omega', 'real_part', 'imaginary_part'))
    for key, value in sorted(summed.items()):
        writer.writerow((*key, value.real, value.imag))
print(f'Wrote {output_csv}')


## Plot a site pair
Choose `(center_site, site_index)` below. The plots show real and imaginary parts of the sum, not its magnitude.

In [ ]:
with output_csv.open(newline='', encoding='utf-8') as f:
    rows = list(csv.DictReader(f))
pairs = sorted({(int(r['center_site']), int(r['site_index'])) for r in rows})
print('Available site pairs:', pairs)
site_pair = pairs[0]  # Change to e.g. (4, 4)
selected = [r for r in rows if (int(r['center_site']), int(r['site_index'])) == site_pair]
if not selected:
    raise ValueError(f'No data for site pair {site_pair}')
times = np.array(sorted({float(r['probe_time']) for r in selected}))
frequencies = np.array(sorted({float(r['omega']) for r in selected}))
plot_frequencies = frequencies - frequency_shift
# Nyquist bound from both time axes of each selected input; requires uniform grids.
spacings = []
for path, _ in inputs:
    with path.open(newline='', encoding='utf-8') as f:
        input_rows = list(csv.DictReader(f))
    for axis in ('t', 't_prime'):
        grid = np.array(sorted({float(row[axis]) for row in input_rows}))
        differences = np.diff(grid)
        if len(differences) == 0 or not np.allclose(differences, differences[0], rtol=1e-5, atol=1e-9):
            raise ValueError(f'{path}: {axis} must be uniformly sampled to define a Nyquist limit')
        spacings.append(differences.max())
nyquist = np.pi / max(spacings)
display_min = max(frequencies.min(), -nyquist) - frequency_shift
display_max = min(frequencies.max(), nyquist) - frequency_shift
if display_min >= display_max:
    raise ValueError('Computed frequency range does not overlap the Nyquist range')
print(f'Nyquist limit: ±{nyquist:g}; displayed shifted range: [{display_min:g}, {display_max:g}]')
values = {(float(r['probe_time']), float(r['omega'])): complex(float(r['real_part']), float(r['imaginary_part'])) for r in selected}
z = np.array([[values[(t, w)] for t in times] for w in frequencies])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, component, title in zip(axes, (z.real, z.imag), ('Real part', 'Imaginary part')):
    mesh = ax.pcolormesh(times, plot_frequencies, component, shading='nearest', cmap='RdBu_r')
    fig.colorbar(mesh, ax=ax, label=title)
    ax.set(xlabel='Probe time', ylabel=f'Frequency ω − {frequency_shift:g}', title=f'{title}: sites {site_pair}')
    ax.set_ylim(display_min, display_max)
plt.show()


## Frequency cut at probe time 8
The nearest sampled time is used and shown in the title.

In [ ]:
target_probe_time = 8.0
index = np.abs(times - target_probe_time).argmin()
actual_time = times[index]
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(plot_frequencies, z[:, index].real, label='Real part')
ax.plot(plot_frequencies, z[:, index].imag, label='Imaginary part')
ax.set(xlabel=f'Frequency ω − {frequency_shift:g}', ylabel='Transform', title=f'Sites {site_pair}, probe time = {actual_time:g} (requested {target_probe_time:g})')
ax.set_xlim(display_min, display_max)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()


## Relative-time transform at fixed probe time
Here T = t_prime = probe time and τ = t − T: F(T, ω) = ∫dt exp(±iω(t−T)) C(t,T). This is a single-time integral, without a Gaussian window or Fourier normalization. Between sampled t_prime values, C is linearly interpolated; extrapolation is rejected. The two-time Gaussian-probe transform above remains separate. Run the preceding cells first.

In [ ]:
from fourier_relative_time import transform_inputs
relative_output_csv = Path('fourier_relative_time_output.csv')
if relative_output_csv.resolve() == output_csv.resolve() or any(relative_output_csv.resolve() == path.resolve() for path, _ in inputs):
    raise ValueError('Relative output must not overwrite another output or an input')
relative_values = transform_inputs(inputs, times, frequencies, input_convention)
with relative_output_csv.open('w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(('center_site', 'site_index', 'probe_time', 'omega', 'real_part', 'imaginary_part'))
    for key, value in sorted(relative_values.items()):
        writer.writerow((*key, value.real, value.imag))
print(f'Wrote {relative_output_csv}')
relative_z = np.array([[relative_values[(*site_pair, t, w)] for t in times] for w in frequencies])
fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
for ax, component, title in zip(axes, (relative_z.real, relative_z.imag), ('Real part', 'Imaginary part')):
    mesh = ax.pcolormesh(times, plot_frequencies, component, shading='nearest', cmap='RdBu_r')
    fig.colorbar(mesh, ax=ax, label=title)
    ax.set(xlabel='Probe time T = t′', ylabel=f'Frequency ω − {frequency_shift:g}', title=f'Relative-time {title}: sites {site_pair}')
    ax.set_ylim(display_min, display_max)
plt.show()
fig, ax = plt.subplots(figsize=(7, 4), constrained_layout=True)
ax.plot(plot_frequencies, relative_z[:, index].real, label='Real part')
ax.plot(plot_frequencies, relative_z[:, index].imag, label='Imaginary part')
ax.set(xlabel=f'Frequency ω − {frequency_shift:g}', ylabel='Relative-time transform', title=f'Sites {site_pair}, T = {actual_time:g}')
ax.set_xlim(display_min, display_max)
ax.legend()
ax.grid(True, alpha=0.3)
plt.show()
